# Training tomato disease

In [ ]:
#mouiting drive to colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#copying data from drive into colab local for speed
!cp -r /content/drive/MyDrive/plantvillage /content/

In [ ]:
from pathlib import Path

dataset_path = Path('/content/plantvillage')

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

below code for loading data set and spliting it into training and  testing

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="both",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

train_ds, val_ds = dataset
class_names = train_ds.class_names

Found 14530 files belonging to 10 classes.
Using 11624 files for training.
Using 2906 files for validation.


This time to optimize pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Data augmentation because our dataset is imbalanced

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

# Loading mobilenetV2 \ This will load pretraind ImageNet \ Freez feture extraction

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
#building Full model

inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

Now i am adjusting and computing class weight minority classes for exam having 299 images will get higher weight

In [ ]:
train_labels = []

for _, labels in train_ds:
    train_labels.extend(labels.numpy())

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = dict(enumerate(class_weights_array))

print(class_weights)

{0: np.float64(0.8484671532846715), 1: np.float64(1.7910631741140215), 2: np.float64(0.9427412814274129), 3: np.float64(1.8900813008130082), 4: np.float64(1.0232394366197184), 5: np.float64(1.1049429657794676), 6: np.float64(1.2915555555555556), 7: np.float64(0.33978368897983047), 8: np.float64(4.884033613445379), 9: np.float64(1.1508910891089108)}


early stoped is true it means it can stop trainig earlier after 12 epoch ᴬ we did this to reduce overfiting when validation stops it will stop furtur training

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/best_tomato_model.keras',
    monitor='val_accuracy',
    save_best_only=True
)

Now compiling model means

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
#final training
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stopping, checkpoint]
)

Epoch 1/15
364/364 ━━━━━━━━━━━━━━━━━━━━ 53s 76ms/step - accuracy: 0.6753 - loss: 1.1002 - val_accuracy: 0.8145 - val_loss: 0.5598
Epoch 2/15
364/364 ━━━━━━━━━━━━━━━━━━━━ 71s 68ms/step - accuracy: 0.8149 - loss: 0.6450 - val_accuracy: 0.8713 - val_loss: 0.4120
Epoch 3/15
364/364 ━━━━━━━━━━━━━━━━━━━━ 36s 70ms/step - accuracy: 0.8410 - loss: 0.5401 - val_accuracy: 0.8775 - val_loss: 0.3893
Epoch 4/15
364/364 ━━━━━━━━━━━━━━━━━━━━ 37s 72ms/step - accuracy: 0.8544 - loss: 0.5000 - val_accuracy: 0.8844 - val_loss: 0.3598
Epoch 5/15
364/364 ━━━━━━━━━━━━━━━━━━━━ 35s 68ms/step - accuracy: 0.8623 - loss: 0.4759 - val_accuracy: 0.8758 - val_loss: 0.3629
Epoch 6/15
364/364 ━━━━━━━━━━━━━━━━━━━━ 36s 70ms/step - accuracy: 0.8634 - loss: 0.4573 - val_accuracy: 0.8816 - val_loss: 0.3355
Epoch 7/15
364/364 ━━━━━━━━━━━━━━━━━━━━ 35s 69ms/step - accuracy: 0.8697 - loss: 0.4408 - val_accuracy: 0.8844 - val_loss: 0.3421
Epoch 8/15
364/364 ━━━━━━━━━━━━━━━━━━━━ 36s 70ms/step - accuracy: 0.8724 - loss: 0.4314 - 

In [ ]:
#saving the best model
best_model = tf.keras.models.load_model('/content/drive/MyDrive/best_tomato_model.keras')

In [ ]:
#generating predictions

y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = best_model.predict(images, verbose=0)
    predicted_labels = tf.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels.numpy())

In [ ]:
#getting classification report
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=class_names))

                                               precision    recall  f1-score   support

                      Tomato___Bacterial_spot       0.84      0.95      0.89       333
                        Tomato___Early_blight       0.92      0.56      0.69       151
                         Tomato___Late_blight       0.89      0.93      0.91       294
                           Tomato___Leaf_Mold       0.85      0.90      0.88       146
                  Tomato___Septoria_leaf_spot       0.90      0.86      0.88       281
Tomato___Spider_mites Two-spotted_spider_mite       0.87      0.90      0.88       289
                         Tomato___Target_Spot       0.78      0.83      0.80       223
       Tomato___Tomato_Yellow_Leaf_Curl_Virus       1.00      0.96      0.98       865
                 Tomato___Tomato_mosaic_virus       0.98      0.87      0.92        61
                             Tomato___healthy       0.89      0.98      0.94       263

                                     accu